# Data Ingestion

In [1]:
### document datastructure

from langchain_core.documents import Document

In [4]:
# Metadata can be used for applying the filters on the documents.
doc = Document(
    page_content="This is the content of the document.",
    metadata={"source": "example.txt", "author": "John Doe"},
)
doc

Document(metadata={'source': 'example.txt', 'author': 'John Doe'}, page_content='This is the content of the document.')

In [5]:
## create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [6]:
sample_texts={
    "../data/text_files/sumo_intro.txt":"""INTRODUCTION TO SUMO WRESTLING

Sumo is Japan's traditional form of wrestling and one of the oldest organized sports still practiced today. More than just a combat sport, sumo combines athletic competition, religious traditions, rituals, discipline, and centuries of history.

The objective is simple:

• Force your opponent out of the circular ring (called a dohyo).
• Or make any part of their body other than the soles of their feet touch the ground.

A match often lasts only a few seconds, but the preparation, strategy, and ritual surrounding it can take years to master.

--------------------------------------------------

A BRIEF HISTORY

Sumo traces its roots back over 1,500 years and has strong connections to Japan's indigenous religion, Shinto.

Originally, sumo was performed as a ritual to:
• Pray for good harvests.
• Entertain the gods.
• Demonstrate strength and discipline.

Even today, many pre-match ceremonies have religious significance, which is why sumo feels very different from modern combat sports like boxing or MMA.

--------------------------------------------------

THE RING (DOHYO)

The wrestling area is a raised clay platform with a circular ring about 4.55 meters (15 feet) in diameter.

Because the ring is relatively small:
• Positioning becomes critical.
• A tiny mistake can end the match instantly.
• Explosive starts often determine the outcome.

--------------------------------------------------

HOW A MATCH WORKS

Before the match:

1. Wrestlers enter the ring.
2. They perform rituals.
3. Salt is thrown to purify the ring.
4. Wrestlers crouch and stare each other down.

The initial charge is called the tachiai.

Once the match begins:
• There are no weight classes in professional sumo.
• Larger wrestlers often have an advantage.
• Technique, balance, timing, and leverage are equally important.

A wrestler wins if:
• The opponent steps outside the ring.
• The opponent touches the ground with a hand, knee, elbow, or any body part besides the feet.

--------------------------------------------------

THE WRESTLERS (RIKISHI)

Sumo wrestlers are called rikishi.

Professional rikishi:
• Live in training houses called heya.
• Follow strict daily schedules.
• Train for many hours each day.
• Eat specialized meals to gain strength and mass.

A famous dish associated with sumo is chankonabe, a hearty stew rich in protein and calories.

--------------------------------------------------

RANKING SYSTEM

Sumo has a highly structured ranking hierarchy.

The highest rank is:

YOKOZUNA

A Yokozuna is expected not only to win but also to display dignity and exemplary conduct.

Other major ranks include:
• Ozeki
• Sekiwake
• Komusubi
• Maegashira

Unlike many sports, a Yokozuna cannot be demoted. If performance declines, retirement is generally expected.

--------------------------------------------------

GRAND TOURNAMENTS

Professional sumo revolves around six major tournaments each year, known as Honbasho.

Each tournament lasts 15 days.

A wrestler typically fights:
• One bout per day.
• Fifteen bouts total.

The wrestler with the best record wins the championship.

--------------------------------------------------

WHY SUMO IS FASCINATING

What makes sumo unique is the contrast between:
• Ancient ritual and modern competition.
• Massive athletes and surprisingly agile movement.
• Matches that may last only seconds but are decided by years of training.

Many newcomers expect sumo to be purely about size, but experienced fans quickly discover that balance, footwork, timing, psychology, and technique often determine the winner.

--------------------------------------------------

FAMOUS SUMO WRESTLERS

Some legendary figures in sumo history include:

• Taiho
• Chiyonofuji Mitsugu
• Hakuho Sho

Many consider Hakuho the greatest sumo wrestler of the modern era due to his extraordinary championship record.

--------------------------------------------------

CONCLUSION

Sumo is far more than a wrestling sport. It is a living piece of Japanese culture that blends athletic excellence, tradition, discipline, and ceremony. While the rules are simple enough to understand in minutes, mastering the sport requires years of dedication, making sumo one of the most unique and fascinating sporting traditions in the world."""
}

In [7]:
for filepath, content in sample_texts.items():
    with open(filepath,'w', encoding= "utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


In [5]:
# Read this particular text with text loader.
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/sumo_intro.txt", encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/sumo_intro.txt'}, page_content="INTRODUCTION TO SUMO WRESTLING\n\nSumo is Japan's traditional form of wrestling and one of the oldest organized sports still practiced today. More than just a combat sport, sumo combines athletic competition, religious traditions, rituals, discipline, and centuries of history.\n\nThe objective is simple:\n\n• Force your opponent out of the circular ring (called a dohyo).\n• Or make any part of their body other than the soles of their feet touch the ground.\n\nA match often lasts only a few seconds, but the preparation, strategy, and ritual surrounding it can take years to master.\n\n--------------------------------------------------\n\nA BRIEF HISTORY\n\nSumo traces its roots back over 1,500 years and has strong connections to Japan's indigenous religion, Shinto.\n\nOriginally, sumo was performed as a ritual to:\n• Pray for good harvests.\n• Entertain the gods.\n• Demonstrate strength and discipline.\n\nE

In [8]:
## Loding with the help of directory loader
from langchain_community.document_loaders import DirectoryLoader

# THis can be used to load different type of files like pdfs and all.
dir_loader = DirectoryLoader("../data/text_files", loader_cls=TextLoader ,glob="*.txt",loader_kwargs={"encoding": "utf-8"}, show_progress=True)

documents = dir_loader.load()
print(documents)

100%|██████████| 1/1 [00:00<00:00, 1335.34it/s]

[Document(metadata={'source': '../data/text_files/sumo_intro.txt'}, page_content="INTRODUCTION TO SUMO WRESTLING\n\nSumo is Japan's traditional form of wrestling and one of the oldest organized sports still practiced today. More than just a combat sport, sumo combines athletic competition, religious traditions, rituals, discipline, and centuries of history.\n\nThe objective is simple:\n\n• Force your opponent out of the circular ring (called a dohyo).\n• Or make any part of their body other than the soles of their feet touch the ground.\n\nA match often lasts only a few seconds, but the preparation, strategy, and ritual surrounding it can take years to master.\n\n--------------------------------------------------\n\nA BRIEF HISTORY\n\nSumo traces its roots back over 1,500 years and has strong connections to Japan's indigenous religion, Shinto.\n\nOriginally, sumo was performed as a ritual to:\n• Pray for good harvests.\n• Entertain the gods.\n• Demonstrate strength and discipline.\n\nE

In [12]:
## Now we will continue to the chunkings ( data ingestion to vector db pipeline)
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [32]:
## read the text file in the directory
def processAllPdfFiles(pdf_directory):
    '''Process all PDF files in the specified directory'''
    all_docs = []

    pdfDir = Path(pdf_directory)

    pdf_files = list(pdfDir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf files in the directory.")

    for file in pdf_files:
        print(f"\nProcessing the {file.name}")

        loader = PyPDFLoader(str(file));
        documents = loader.load()

        for doc in documents:
            doc.metadata["source_file"] = file.name
            doc.metadata["file_type"] = 'pdf'
        
        all_docs.extend(documents)

    return all_docs



In [33]:
all_txt = processAllPdfFiles("../data/pdf_files")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 106 0 (offset 0)
Ignoring wrong pointing object 232 0 (offset 0)


Found 2 pdf files in the directory.

Processing the httpslms.bennett.edu.inpluginfile.php352044mod_resourcecontent1GAN.pdf 3.pdf

Processing the AadityaBirSingh_2.pdf


In [34]:
all_txt

[Document(metadata={'producer': 'iOS Version 18.4.1 (Build 22E252) Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20250513181325Z00'00'", 'title': 'GAN Final.pptx', 'moddate': "D:20250513181325Z00'00'", 'source': '../data/pdf_files/httpslms.bennett.edu.inpluginfile.php352044mod_resourcecontent1GAN.pdf 3.pdf', 'total_pages': 73, 'page': 0, 'page_label': '1', 'source_file': 'httpslms.bennett.edu.inpluginfile.php352044mod_resourcecontent1GAN.pdf 3.pdf', 'file_type': 'pdf'}, page_content='Generative\tAdversarial\t\nNetworks\t(GANs)\nFromIan\tGoodfellow\tet\tal.\nA\tshort\ttutorial\tby\t:-\nBinglin,\tShashank\t&Bhargav'),
 Document(metadata={'producer': 'iOS Version 18.4.1 (Build 22E252) Quartz PDFContext', 'creator': 'PowerPoint', 'creationdate': "D:20250513181325Z00'00'", 'title': 'GAN Final.pptx', 'moddate': "D:20250513181325Z00'00'", 'source': '../data/pdf_files/httpslms.bennett.edu.inpluginfile.php352044mod_resourcecontent1GAN.pdf 3.pdf', 'total_pages': 73, 'page': 1, 

In [35]:
## chunking method 

def splitDocs(documents, chunk_size = 1000, chunk_overlap = 200):
    """ Split docs into small chunks for better RAG """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ",""]
    )

    split_docs = text_splitter.split_documents(documents)

    return split_docs

In [36]:
chunks = splitDocs(documents)
chunks

[Document(metadata={'source': '../data/text_files/sumo_intro.txt'}, page_content="INTRODUCTION TO SUMO WRESTLING\n\nSumo is Japan's traditional form of wrestling and one of the oldest organized sports still practiced today. More than just a combat sport, sumo combines athletic competition, religious traditions, rituals, discipline, and centuries of history.\n\nThe objective is simple:\n\n• Force your opponent out of the circular ring (called a dohyo).\n• Or make any part of their body other than the soles of their feet touch the ground.\n\nA match often lasts only a few seconds, but the preparation, strategy, and ritual surrounding it can take years to master.\n\n--------------------------------------------------\n\nA BRIEF HISTORY\n\nSumo traces its roots back over 1,500 years and has strong connections to Japan's indigenous religion, Shinto.\n\nOriginally, sumo was performed as a ritual to:\n• Pray for good harvests.\n• Entertain the gods.\n• Demonstrate strength and discipline."),
 

In [37]:
## Now we will be focusing on the embeddings
import numpy
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Dict, Tuple, Any, List
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
# Try to understand this modular code tomorrow or whenever you are free.
class EmbeddingManager:
    """ Handles document embedding using the SentenceTransformer """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        
        """Initiallize the embedding manaager
           Args:
            model_name (str, optional): The name of the SentenceTransformer model to use. Defaults to "all-MiniLM-L6-v2".
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully with dimensions {self.model.get_sentence_embedding_dimension()}.")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> numpy.ndarray:
        """Generate embeddings for a list of texts.
           Args:
            texts (List[str]): A list of strings to generate embeddings for.
           Returns:
            numpy.ndarray: An array of embeddings corresponding to the input texts.
        """
        if not self.model:
            raise ValueError("Model is not loaded.")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise


embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12822.43it/s]


Model 'all-MiniLM-L6-v2' loaded successfully with dimensions 384.


/var/folders/c7/694hx7gj2sbgr7v9n3vy8py40000gn/T/ipykernel_89786/2626011096.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model '{self.model_name}' loaded successfully with dimensions {self.model.get_sentence_embedding_dimension()}.")


## Converting Documents to Vector Embeddings

Pipeline:
1. Load documents → split into chunks
2. Extract `page_content` from each chunk
3. Pass texts to `EmbeddingManager.generate_embeddings()`
4. Inspect the resulting embedding matrix

In [ ]:
# Step 1: Load and chunk the sumo text document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("../data/text_files/sumo_intro.txt", encoding="utf-8")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(raw_docs)

print(f"Total chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {len(chunk.page_content)} chars")

In [ ]:
# Step 2: Extract plain text from each chunk
texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]

print(f"Texts extracted: {len(texts)}")
print(f"\nSample text (chunk 0):\n{texts[0][:200]}...")

In [ ]:
# Step 3: Generate embeddings for all chunks
embeddings = embedding_manager.generate_embeddings(texts)

print(f"\nEmbedding matrix shape : {embeddings.shape}")
print(f"  → {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions")
print(f"\nFirst embedding (first 10 values):\n{embeddings[0][:10]}")

In [ ]:
# Step 4: Bundle chunks with their embeddings for downstream use
embedded_docs = [
    {
        "chunk_id": i,
        "text": texts[i],
        "metadata": metadatas[i],
        "embedding": embeddings[i],
    }
    for i in range(len(chunks))
]

print(f"Embedded {len(embedded_docs)} documents")
print(f"\nSample entry keys: {list(embedded_docs[0].keys())}")
print(f"Source: {embedded_docs[0]['metadata']['source']}")
print(f"Embedding dtype: {embedded_docs[0]['embedding'].dtype}")

### Quick Similarity Search (sanity check)

Embed a query and rank chunks by cosine similarity to verify the embeddings are meaningful.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "What is the highest rank in sumo?"
query_embedding = embedding_manager.generate_embeddings([query])  # shape (1, 384)

scores = cosine_similarity(query_embedding, embeddings)[0]  # shape (num_chunks,)
ranked = np.argsort(scores)[::-1]

print(f"Query: {query}\n")
print("Top 3 most relevant chunks:")
for rank, idx in enumerate(ranked[:3], 1):
    print(f"\n  [{rank}] Chunk {idx} | score={scores[idx]:.4f}")
    print(f"      {texts[idx][:300]}...")